<h1>Implementation of the Pachner 2-2 move</h1>

The method used to implement the Pachner 2-2 move in `trivalent` can be confusing, so here I spell out the method used in the program, and give examples.

We will outline the basic procedure to start, then go through two examples of the 2-2 move in action -- the 3-prism graph mapping to itself, and the 4-prism mapping to the standard graph $\Delta_8$. Before we begin, we import a couple of necessary modules.

In [1]:
from copy import deepcopy
from trivalent import Graph

The following procedure will be used later to convert edge permutation bitmasks into a list of signed edge indices. The sign is whether the permutation takes the edge to the same canonical direction ($+1$) or opposite direction ($-1$) as the target edge. To avoid having the ambiguity of a sign on the label $0$, we increase the edge label by 1.

In [2]:
def unpackPermBits(bits, num_edges):
    b = max(1, (2 * num_edges).bit_length())
    mask = (1 << b) - 1
    
    perm = [None] * num_edges
    for idx in range(num_edges):
        perm_idx = ((bits >> (idx * b)) & mask)
        if perm_idx > num_edges:
            perm_idx = -(perm_idx - num_edges)
        perm[idx] = perm_idx
    
    return perm

Now to an overview of the 2-2 Pachner move. Suppose we start with a source edge $xy$, for vertices $x, y$, where the cyclic order around $x$ is $aby$, and the cyclic order around $y$ is $cdx$, as shown below.

![](initial-edge-config.png)

We assume that $x < y$, i.e. $x$ has a lower vertex label. We want to rewire the graph so that afterwards, the (new) vertex $x$ has cyclic order $bcy$, while the (new) vertex $y$ has cyclic order $dax$.

![](final-edge-config.png)

To do this, we make the following changes in the graph edge list:

1. Change edge $ax$ to $ay$.
2. Change edge $cy$ to $cx$.
3. Change the location of $xy$ (now the *new* edge) so that $x, y$ have the appropriate cyclic orders without modifying the cyclic orders of $a, b, c, d$.

This last part is a bit intricate, so we give some examples of the process. We will show how to map both the edges and faces in the graphs before and after the Pachner move. We also highlight a few subtleties in how this is done, in the permutations used for the given mappings.

<h2>Example 1: the 3-prism graph</h2>

<h3>Mechanics of the 2-2 move</h3>

We use the 3-prism as a first example; the 2-2 move takes this graph to itself, so the edges and faces are permuted amongst themselves. We will act on the `01` edge with the 2-2 move, so given below is the picture of the action on the entire graph. After the Pachner move, we redraw the graph to put it back into its canonical shape for easier interpretation.

![](Pachner-move-1.png)

Here is a table giving the internal vertices, and their immediate neighbors, matching the previous discussion.

| x | y | a | b | c | d |
|---|---|---|---|---|---|
| 0 | 1 | 4 | 5 | 2 | 3 |

We create a `trivalent.Graph` object based on this edge list, and find the faces of this graph for later use. In this particular form of the edge list, the specific edges affected by the 2-2 move are given in the order `xy ax bx cy ... dy`. This ordering will be relevant when trying to decide where to place `xy` after the move.

In [3]:
numVert = 6
numEdges = 3 * numVert // 2

# 3-prism

G = Graph(numVert)
G.addEdges([[0, 1], [0, 4], [0, 5], [1, 2], [3, 4], [4, 5], [2, 5], [1, 3], [2, 3]])
G.findFaces()

The `01` edge is the first edge in the edge list, so in `pachner22()`, we give the edge index 0 as the argument to define the 2-2 move. The procedure returns the edge index `edgeIdx` of the graph edge resulting from the Pachner move in the modified edge list, as well as the signed edge permutation bitmask `edgePerm`. We do not care about the possible orientation changes here, so we drop the signs and write the edge indices in `edgePerm` (and subsequent permutations) as 0-based.

The 2-2 move is executed on a copy `H` of the original graph, so that running the cell below twice does not lead to an error.

In [4]:
H = deepcopy(G)

edgeIdx, edgePermBits = H.pachner22(0)

edgePerm = [abs(perm) - 1 for perm in unpackPermBits(edgePermBits, numEdges)]
print(edgePerm)

[1, 0, 2, 3, 4, 5, 6, 7, 8]


So let's figure out what has happened with the edge list after the 2-2 move. Remember we are starting with the edge list `xy ax bx cy ... dy`. Using the method outlined above, we have the following changes:

1. Change $ax$ to $ay$: `xy ay bx cy ... dy`
2. Change $cy$ to $cx$: `xy ay bx cx ... dy`

At this point, the cyclic order for $x$ is now $ybc$, and for $y$ is $adx$. Notice that this is correct (up to cyclic order) for $x$, but not for $y$ -- it should be $dax$. The simplest way to fix this is to move $xy$ in the edge list, so that both cyclic orders are correct.

In order to have the correct cyclic orders, $xy$ must appear either before $bx$ or after $cx$ in the edge list, while it must also appear before $dy$ or after $ay$. These conditions are met if, for example, we swap edges $xy$ and $ay$ in the ordering, to give `ay xy bx cx ... dy`. This is exactly what is done by `pachner22()`, which looks for the lowest index in the graph edge list giving consistent results. Note that `04` ($ax$) has been changed to `14` ($ay$), and `12` ($cy$) is now `02` ($cx$).

In [5]:
print(f'edge list after 2-2 move: {" ".join([str(edge.start) + str(edge.end) for edge in H.edgeList()])}')

edge list after 2-2 move: 14 01 05 02 34 45 25 13 23


It should be noted that this explicit edge list matches the edge index list of the permutation `edgePerm` given previously, mapping the edge list of `H` before and after the 2-2 move, when one factors in the changes $ax \to ay, cy \to cx$.

<h3> Edge permutations</h3>

In order to map these faces back to the canonical form of the 3-prism graph, we need to establish the graph isomorphism between `H` and the original `G`. In `trivalent`, the `__eq__` method returns a bitmask of the signed permutation between the two graphs, when the two are isomorphic (otherwise, it returns `False`).

Note that changing the ordering of the graphs in `__eq__` will give a different permutation.

In [6]:
# Mapping edges in G -> edges in H

GH_permBits = (G == H)
GH_perm = [abs(perm) - 1 for perm in unpackPermBits(GH_permBits, numEdges)]

print(f'map G -> H: {GH_perm}')

 # Mapping edges in H -> edges in G

HG_permBits = (H == G)
HG_perm = [abs(perm) - 1 for perm in unpackPermBits(HG_permBits, numEdges)]

print(f'map H -> G: {HG_perm}')

map G -> H: [1, 2, 3, 7, 5, 6, 8, 0, 4]
map H -> G: [1, 4, 7, 8, 2, 0, 3, 5, 6]


Combining the permutations does not give the identity permutation, although it is a symmetry of the graph. Specifically, on graph `G` it is an inversion preserving the edge $e_{25}$ (but reversing its orientation). Think of the 3-prism graph as the 2d projection of a cylinder, with the face $123$ as the facing side; the inversion would result when the cylinder is spun on an axis through the side edge $e_{25}$.  We shall return to this point below.

In [7]:
print(f'map G -> G: {[HG_perm[GH_perm[idx]] for idx in range(numEdges)]}')

map G -> G: [4, 7, 8, 5, 0, 3, 6, 1, 2]


First, we address where the permutation `GH_perm` is coming from. In the picture below, the 2-2 move is redrawn, with the addition of the edges labeled by their index in the graph edge list. Notice that the *same* edges have different labels before and after the move, since their ordering in the edge list has changed, e.g. $e_{01}$ was the index 0 edge for `G`, but is now the index 1 edge for `H`.

![](Pachner-move-2.png)

The permutation `GH_perm` from graph `G` to graph `H`, and the permutation `HG_perm` working in the opposite direction, are found by searching for maps between the edge lists that explicitly give the isomorphism. Because the structure of the two edge lists are different, this isomorphism will be non-trivial. Specifically, note that vertex 0 appears in all the first three edges of `G`, but no vertex does the same in the `H` edge list.

Starting with `G`, the first edge that has a similar structure to $e_{01}$ is the index 1 edge with the same labels in `H`. Checking the case where the two edges have the same canonical orientation gives the `G` $\to$ `H` mapping $e_{04} \to e_{05}, e_{05} \to e_{02}, e_{13} \to e_{14}$, and $e_{12} \to e_{13}$. Note this has nothing to do with the specific way the 2-2 move was done; `__eq__` is only looking for equivalent structures in the two graphs. The rest of the isomorphism follows from these associations:

| G | $e_{01}$ | $e_{04}$ | $e_{05}$ | $e_{12}$ | $e_{34}$ | $e_{45}$ | $e_{25}$ | $e_{13}$ | $e_{23}$ |
|---|----------|----------|----------|----------|----------|----------|----------|----------|----------|
| H | $e_{01}$ | $e_{05}$ | $e_{02}$ | $e_{13}$ | $e_{45}$ | $e_{25}$ | $e_{23}$ | $e_{14}$ | $e_{34}$ |

The permutation `GH_perm` is giving these edges in terms of their indices in the respective edge lists for `G` and `H`.

| G | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| H | 1 | 2 | 3 | 7 | 5 | 6 | 8 | 0 | 4 |

For the reverse process, the first edge in `G` that is similar to the index 0 edge $e_{14}$ in `H` is the index 1 edge $e_{04}$. The fact that these two edges can be consistently associated is why the combination of the two edge permutations gives an inversion, since one is on the "outside" of the graph and the other is on the "inside". However, it is also why they cannot be mapped in the same direction, i.e. specifically associating `G` vertex 0 with `H` vertex 1, and `G` vertex 4 with `H` vertex 4. If this were done, then vertex 5 in graph `G` would be mapped to both vertices 0 and 5 in graph `H`. Reversing their orientations between the two graphs is done by an inversion perserving the location (but not the vertex labels or orientation) of edge $e_{25}$, as described previously. This means that the neighboring edges are associated by $e_{13} \to e_{45}, e_{01} \to e_{34}, e_{34} \to e_{05}$, and $e_{13} \to e_{45}$. The rest of the mapping is given below.

| H | $e_{14}$ | $e_{01}$ | $e_{05}$ | $e_{02}$ | $e_{34}$ | $e_{45}$ | $e_{25}$ | $e_{13}$ | $e_{23}$ |
|---|----------|----------|----------|----------|----------|----------|----------|----------|----------|
| G | $e_{04}$ | $e_{34}$ | $e_{13}$ | $e_{23}$ | $e_{05}$ | $e_{01}$ | $e_{12}$ | $e_{45}$ | $e_{25}$ |

| H | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| G | 1 | 4 | 7 | 8 | 2 | 0 | 3 | 5 | 6 |

<h3>Face permutations</h3>

Now we turn to the permutations of the graph faces. We can go back to the object `G`, and print out its faces, either in terms of the boundary edges or vertices.

In [8]:
print('Face edges with vertex labels:')
for faceNum, face in enumerate(G.faceList()):
    print(f'\tface {faceNum}: ' + ' '.join([str(edge.start) + str(edge.end) for edge in face.edgeList()]))

print('\nFace edges by edge list index:')
for faceNum, face in enumerate(G.faceList()):
    print(f'\tface {faceNum}: ' + ''.join([str(G.edgeList().index(edge)) for edge in face.edgeList()]))

Face edges with vertex labels:
	face 0: 05 25 12 01
	face 1: 25 45 34 23
	face 2: 45 05 04
	face 3: 34 04 01 13
	face 4: 23 13 12

Face edges by edge list index:
	face 0: 2630
	face 1: 6548
	face 2: 521
	face 3: 4107
	face 4: 873


Doing the same thing with the graph `H` transformed by the 2-2 move, note the ordering of the faces has changed -- specifically, the faces with the same size do not appear in the same order as in the list given above for `G`. The 2-2 move increases two faces by one edge, and decreases two other faces by one edge as well, which permutes the faces in the graph.

In [9]:
print('Face edges with vertex labels:')
for faceNum, face in enumerate(H.faceList()):
    print(f'\tface {faceNum}: ' + ' '.join([str(edge.start) + str(edge.end) for edge in face.edgeList()]))

print('\nFace edges by edge list index:')
for faceNum, face in enumerate(H.faceList()):
    print(f'\tface {faceNum}: ' + ''.join([str(H.edgeList().index(edge)) for edge in face.edgeList()]))

Face edges with vertex labels:
	face 0: 05 25 02
	face 1: 25 45 34 23
	face 2: 45 05 01 14
	face 3: 34 14 13
	face 4: 23 13 01 02

Face edges by edge list index:
	face 0: 263
	face 1: 6548
	face 2: 5210
	face 3: 407
	face 4: 8713


With the edge permutations given above, we can simply map from one set of faces to the other. Note this is only possible because the underlying graphs for both `G` and `H` are the same. Starting with the graph `G` and permutation `GH_perm`, we get

| G    | H    |
|------|------|
| 2630 | 3871 |
| 6548 | 8654 |
| 521  | 632  |
| 4107 | 5210 |
| 873  | 407  |

Up to cyclic permutation of the edge ordering for each face, this gives exactly the faces of graph `H`. In particular, the faces are permuted as

| G | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| H | 4 | 1 | 0 | 2 | 3 |

If we begin with graph `H` and permutation `HG_perm`, then the results are as below.

| H    | G    |
|------|------|
| 263  | 738  |
| 6548 | 3026 |
| 5210 | 0741 |
| 407  | 215  |
| 8713 | 6548 |

| H | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| G | 4 | 0 | 3 | 2 | 1 |

Applying both permutations gives the faces of graph `G` permuting as $f_0 \leftrightarrow f_1, f_2 \leftrightarrow f_4$, and $f_3$ mapping to itself. As previously discussed, this is the product of an inversion, so the orientations of the faces would change, although we are not concerned with that here.

<h3>Putting it together</h3>

To set up for going through the same process for two different graph structures, let's go through what we have done above. First, we started with the 3-prism graph `G`, and used the 2-2 move on $e_{01}$, giving the graph `H`. The procedure `pachner22()` found the permutation `edgePerm` that mapped from the edge list of `G` to that of `H`. Next, we wanted to find an isomorphism between `H` back to `G`, which was `HG_perm`, from using `(H == G)`. We should be able to combine these two permutations, and get the same overall result.

In [10]:
print([HG_perm[edgePerm[idx]] for idx in range(numEdges)])

[4, 1, 7, 8, 2, 0, 3, 5, 6]


Indeed, if we start with the edge list for `G`, and follow both the 2-2 move and the subsequent isomorphism, this is exactly what we get -- it is the same as the map `H` $\to$ `G` listed above, save for the transposition of the first two elements, the result of the Pachner move on the original `G` edge list to get `H`.

It is important to realize that the graph isomorphism algorithm starts by looking for analogous edges in the graphs being compared. Thus, the edge permutations resulting from `(G == H)` and `(H == G)`, when multiplied together, will generically give a non-trivial automorphism of the 3-prism graph. One way to see this is to construct different descriptions of the graph, by permuting the order of the edge list. Here is an example, giving another inversion, this time preserving the edge $e_{34}$.

In [11]:
# In canonical edge list, permute one edge from end of list to beginning

G_p = Graph(numVert)
G_p.addEdges([[2, 3], [0, 1], [0, 4], [0, 5], [1, 2], [3, 4], [4, 5], [2, 5], [1, 3]])

# Define two ways to map between the graphs

alt_permBits = (G_p == G)
alt_perm_1 = [abs(perm) - 1 for perm in unpackPermBits(alt_permBits, numEdges)]
print(f'edge permutation G_p -> G: {alt_perm_1}')

alt_permBits = (G == G_p)
alt_perm_2 = [abs(perm) - 1 for perm in unpackPermBits(alt_permBits, numEdges)]
print(f'edge permutation G -> G_p: {alt_perm_2}')

# Show the permutation product is non-trivial

print(f'permutation product: {[alt_perm_2[alt_perm_1[idx]] for idx in range(numEdges)]}')

edge permutation G_p -> G: [1, 6, 8, 3, 2, 4, 7, 0, 5]
edge permutation G -> G_p: [1, 2, 3, 4, 5, 6, 7, 8, 0]
permutation product: [2, 7, 0, 4, 3, 5, 8, 1, 6]


<h2>Example 2: From the 4-prism to the standard graph $\Delta_8$</h2>

<h3>The Pachner move and the edge list</h3>

As our second example, we use a 2-2 move that changes the underlying graph structure. Starting with the 4-prism, because of its high degree of symmetry, there is only one non-isomorphic edge. Taking this as the source edge, the Pachner move results in the standard graph $\Delta_8$. Redrawing the graph in the second step will aid in understanding the graph isomorphism results later for the new graph.

![2-2 move on 4-prism graph](Pachner-move-3.png)

Below, we set up the graph `G1` in the same way we began with the 3-prism example.

In [ ]:
numVert = 8
numEdges = 3 * numVert // 2

G1 = Graph(numVert)
G1.addEdges([[4, 6], [0, 6], [0, 5], [0, 7], [2, 7], [1, 7], \
             [3, 4], [4, 5], [2, 5], [1, 3], [1, 6], [2, 3]])
G1.findFaces()

Using the edge $e_{46}$ as our source edge, we have that $a = 3, b = 5, c = 0, d = 4$, so that 

| 46 | 06 | 05 | 07 | 27 | 17 | 34 | 45 | 25 | 13 | 16 | 23 |
|----|----|----|----|----|----|----|----|----|----|----|----|
|$xy$|$cy$|    |    |    |    |$ax$|$bx$|    |    |$dy$|    |

Thus, $cy \to cx$ gives $e_{06} \to e_{04}$, while $ax \to ay$ gives $e_{34} \to e_{36}$. Finally, we see from the ordering that $xy$ must be after $ay$, but before $bx$, giving only a single slot to place the edge.

| 04 | 05 | 07 | 27 | 17 | 36 | 46 | 45 | 25 | 13 | 16 | 23 |
|----|----|----|----|----|----|----|----|----|----|----|----|
|$cx$|    |    |    |    |$ay$|$xy$|$bx$|    |    |$dy$|    |

Working out the required permutation of the edges gives the result returned by `pachner22()` below.

In [13]:
K = deepcopy(G1)

# Edge list for K after 2-2 move

edgeIdx, edgePermBits = K.pachner22(0)
print(f'K edge list: {" ".join([str(edge.start) + str(edge.end) for edge in K.edgeList()])}')

# Permutation of edge list order from Pachner move

edgePerm = [abs(perm) - 1 for perm in unpackPermBits(edgePermBits, numEdges)]
print(f'edge list perm: {edgePerm}')

K edge list: 04 05 07 27 17 36 46 45 25 13 16 23
edge list perm: [6, 0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11]


<h3>Edge permutations</h3>

Now we must map this to the canonical form of the standard graph $\Delta_8$, shown in the figure below.

![Standard graph labeling](delta-8.png)

Using its edge list, we create the corresponding graph `G0`, and then get the necessary edge permutation from `K` to `G0`.

In [14]:
G0 = Graph(numVert)
G0.addEdges([[0, 1], [0, 6], [0, 7], [4, 6], [6, 7], [5, 7], \
             [1, 2], [3, 4], [4, 5], [2, 5], [1, 3], [2, 3]])
G0.findFaces()

# Get edge permutation from graph isomorphism

eqPermBits = (K == G0)
eqPerm = [abs(perm) - 1 for perm in unpackPermBits(eqPermBits, numEdges)]
print(eqPerm)

[2, 4, 5, 8, 9, 10, 0, 1, 3, 11, 6, 7]


When drawing the 2-2 move above, we anticipated what would happen at this step. Consider the edge list for the graph `K` after the Pachner move. The isomorphism algorithm will try to find an edge in `G0` with the same connectivity as $e_{04}$ in `K`. Going through the list of edges for `G0`, $e_{01}$ does not satisfy this requirement. The edge $e_{06}$ has the same problem we saw earlier with the 3-prism: this would require the vertex 7 in `G0` to map to two vertices 6, 7 in `K`. Said another way, the 3-cycle face is on different sides of the edges $e_{04}$ in `K`, and $e_{06}$ in `G0`, so reversing the orientation of one edge will not fix the problem without giving other problems. The next edge $e_{07}$ is the one that works out, as can be seen by the redrawn `K` graph in the diagram above. From this, we have the following edge permutation; we use hexadecimal in the edge index tables for more compact notation.

| $K$ | $e_{04}$ | $e_{05}$ | $e_{07}$ | $e_{27}$ | $e_{17}$ | $e_{36}$ | $e_{46}$ | $e_{45}$ | $e_{25}$ | $e_{13}$ | $e_{16}$ | $e_{23}$ |
|-----|----------|----------|----------|----------|----------|----------|----------|----------|----------|----------|----------|----------|
|$G_0$| $e_{07}$ | $e_{67}$ | $e_{57}$ | $e_{45}$ | $e_{25}$ | $e_{13}$ | $e_{01}$ | $e_{06}$ | $e_{46}$ | $e_{23}$ | $e_{12}$ | $e_{34}$ |

| $K$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | a | b |
|-----|---|---|---|---|---|---|---|---|---|---|---|---|
|$G_0$| 2 | 4 | 5 | 8 | 9 | a | 0 | 1 | 3 | b | 6 | 7 |

This matches `eqPerm` given previously.

Now, if we combine the two edge permutations `edgePerm` (from the 2-2 move) and `eqPerm` (from the isomorphism), we get the combined change in the edge lists from the starting graph `G1` to the final graph `G0`, including the edge changes $ax \to ay, cy \to cx$.

| $G_1$ | $e_{46}$ | $e_{06}$ | $e_{05}$ | $e_{07}$ | $e_{27}$ | $e_{17}$ | $e_{34}$ | $e_{45}$ | $e_{25}$ | $e_{13}$ | $e_{16}$ | $e_{23}$ |
|-------|----------|----------|----------|----------|----------|----------|----------|----------|----------|----------|----------|----------|
| $G_0$ | $e_{01}$ | $e_{07}$ | $e_{67}$ | $e_{57}$ | $e_{45}$ | $e_{25}$ | $e_{13}$ | $e_{06}$ | $e_{46}$ | $e_{23}$ | $e_{12}$ | $e_{34}$ |

| $G_1$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | a | b |
|-------|---|---|---|---|---|---|---|---|---|---|---|---|
| $G_0$ | 0 | 2 | 4 | 5 | 8 | 9 | a | 1 | 3 | b | 6 | 7 |

In [15]:
finalPerm = [eqPerm[edgePerm[idx]] for idx in range(numEdges)]
print(finalPerm)

[0, 2, 4, 5, 8, 9, 10, 1, 3, 11, 6, 7]


<h3>Face permutations</h3>

Finally, we can use this permutation to match faces from `G1` to `G0`. We print out the various faces for both graphs.

In [16]:
print('\nG1 face edges by edge list index:')
for faceNum, face in enumerate(G1.faceList()):
    print(f'\tface {faceNum}: ' + ''.join([f"{G1.edgeList().index(edge):x}" for edge in face.edgeList()]))

print('\nG0 face edges by edge list index:')
for faceNum, face in enumerate(G0.faceList()):
    print(f'\tface {faceNum}: ' + ''.join([f"{G0.edgeList().index(edge):x}" for edge in face.edgeList()]))


G1 face edges by edge list index:
	face 0: 35a1
	face 1: 54b9
	face 2: 4328
	face 3: 0a96
	face 4: 1072
	face 5: 876b

G0 face edges by edge list index:
	face 0: 25960
	face 1: 5438
	face 2: 421
	face 3: 310a7
	face 4: 987b
	face 5: ba6


Because the 2-2 move changes the size of four of the faces, the process is not as simple as it was for the 3-prism, where we could merely use the permutation `finalPerm` on the edge labels. We can see what has happened by showing the faces of the intermediate graph `K`.

In [17]:
print('\nK face edges by edge list index:')
for faceNum, face in enumerate(K.faceList()):
    print(f'\tface {faceNum}: ' + ''.join([f"{K.edgeList().index(edge):x}" for edge in face.edgeList()]))


K face edges by edge list index:
	face 0: 24a60
	face 1: 43b9
	face 2: 3218
	face 3: a95
	face 4: 071
	face 5: 8765b


The permutation `edgePerm` allows us to map betwen the initial edge indices of `G1` to those of `K`. So we can fudge the process by using this mapping on the initial edges of `G1`, to get the face edge indices ("pre-move `K`") as if the 2-2 move never happened. Thus, the faces will all remain their same size. After this, we can go one step further, and see how the Pachner move changes the number of boundary edges for each face, giving "actual `K`". The process is shown in the table below.

| $G_1$ | pre-move $K$ | actual $K$ | $G_0$ |
|------:|-------------:|-----------:|------:|
|  35a1 | 24a0         |      24a60 | 59062 |
|  54b9 | 43b9         |       43b9 |  987b |
|  4328 | 3218         |       3128 |  8543 |
|  0a96 | 6a95         |        a95 |   6ba |
|  1072 | 0671         |        071 |   214 |
|  876b | 876b         |      8765b | 310a7 |

From the actual `K` edge indices for each face are obtained, then using the permutation `eqPerm` coming from the graph isomorphism algorithm takes us to a description of the faces in `G0`, i.e. the standard graph $\Delta_8$. This is the last column in the table above. By matching these entries (up to cyclic order) with the `G0` faces listed above, we get our final face permutation.

| $G_1$ | 0 | 1 | 2 | 3 | 4 | 5 |
|-------|---|---|---|---|---|---|
| $G_0$ | 0 | 4 | 1 | 5 | 2 | 3 |